# 04 — PDP / ICE Plot
Effet de `friends_followers_ratio` sur la probabilité prédite d'être un bot.
- **ICE** : une courbe par compte, colorée par label réel
- **PDP** : moyenne de toutes les courbes ICE
- **Scatter** : position réelle des points dans le dataset

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import sys
sys.path.insert(0, '../src')
from data import load_dataset_split, FEATURES

In [ ]:
X_train, X_test, y_train, y_test = load_dataset_split()
model = joblib.load('../models/xgboost.joblib')
print('Dataset test :', X_test.shape)

## Calcul des courbes ICE

In [ ]:
feat = 'friends_followers_ratio'
grid = np.linspace(X_test[feat].quantile(0.05), X_test[feat].quantile(0.95), 60)
print(f'Grille : {grid.min():.2f} → {grid.max():.2f} ({len(grid)} points)')

In [ ]:
idx_bots   = y_test[y_test == 1].sample(150, random_state=42).index
idx_humans = y_test[y_test == 0].sample(150, random_state=42).index
sample_idx = idx_bots.append(idx_humans)
X_sample = X_test.loc[sample_idx].copy()
y_sample  = y_test.loc[sample_idx].values
print(f'Échantillon : {len(X_sample)} comptes ({y_sample.sum()} bots, {(y_sample==0).sum()} humains)')

In [ ]:
ice_curves = np.zeros((len(X_sample), len(grid)))
for j, val in enumerate(grid):
    X_tmp = X_sample.copy()
    X_tmp[feat] = val
    ice_curves[:, j] = model.predict_proba(X_tmp)[:, 1]
print('ICE calculées :', ice_curves.shape)

## PDP / ICE + scatter des points réels

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# ── ICE curves ────────────────────────────────────────────────────────────
for i, label in enumerate(y_sample):
    ax.plot(grid, ice_curves[i],
            color='#EF4444' if label == 1 else '#06B6D4',
            alpha=0.05, linewidth=0.8)

# ── PDP ───────────────────────────────────────────────────────────────────
pdp       = ice_curves.mean(axis=0)
pdp_bot   = ice_curves[y_sample == 1].mean(axis=0)
pdp_human = ice_curves[y_sample == 0].mean(axis=0)

ax.plot(grid, pdp,       color='#F59E0B', linewidth=3.5, label='PDP — effet moyen', zorder=5)
ax.plot(grid, pdp_bot,   color='#EF4444', linewidth=2, linestyle='--', label='PDP bots',    zorder=4)
ax.plot(grid, pdp_human, color='#06B6D4', linewidth=2, linestyle='--', label='PDP humains', zorder=4)

# ── Scatter : position réelle des points ──────────────────────────────────
x_real = X_sample[feat].values
y_proba = model.predict_proba(X_sample)[:, 1]

ax.scatter(x_real[y_sample == 0], y_proba[y_sample == 0],
           color='#06B6D4', alpha=0.45, s=25, edgecolors='white',
           linewidths=0.3, zorder=6, label='Points réels — humains')
ax.scatter(x_real[y_sample == 1], y_proba[y_sample == 1],
           color='#EF4444', alpha=0.45, s=25, edgecolors='white',
           linewidths=0.3, zorder=6, label='Points réels — bots')

# ── Seuil suspect ─────────────────────────────────────────────────────────
ax.axvline(x=5, color='#9CA3AF', linestyle=':', linewidth=1.5, label='Seuil suspect (ratio = 5)')

# ── Légende proxy ICE ─────────────────────────────────────────────────────
handles, _ = ax.get_legend_handles_labels()
handles += [
    Line2D([0],[0], color='#EF4444', alpha=0.35, linewidth=2, label='ICE — bots'),
    Line2D([0],[0], color='#06B6D4', alpha=0.35, linewidth=2, label='ICE — humains'),
]
ax.legend(handles=handles, fontsize=10, loc='upper left')

ax.set_xlabel('friends_followers_ratio (abonnements / followers)', fontsize=12)
ax.set_ylabel('Probabilité prédite — bot', fontsize=12)
ax.set_title('PDP / ICE + scatter — effet de friends_followers_ratio sur la prédiction',
             fontsize=13, fontweight='bold')
ax.set_ylim(-0.05, 1.1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/pdp_ice_friends_followers.png', dpi=150)
plt.show()
print('Sauvegardé : plots/pdp_ice_friends_followers.png')

## Lecture du graphique
- **Points rouges** : bots réels dans le jeu de test — leur position (x = ratio réel, y = proba prédite)
- **Points bleus** : humains réels — proba prédite généralement basse
- **Lignes ICE** : comment la proba évolue si on fait varier le ratio pour ce compte
- **Ligne orange** : PDP global — l'effet moyen
- **Pointillés** : PDP séparé par classe